# 01 · Quick look at one take

Raw waveforms first, before any processing, because a mis-wired or silent channel, a clipped
channel or a knock on the bench is obvious in the raw samples and invisible in a summary number.
Then process that one take (a single streaming pass, cached) and look at its spectrogram.

In [ ]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
TAKE = "DEMO_002"            # a take name from notebook 00
T0, T1 = 39.5, 41.5          # a short window [s] for the raw waveform view

In [ ]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

In [ ]:
takes = za.discover_takes(cfg.data_dir, recursive=cfg.recursive, pattern=cfg.file_pattern)
take = za.find_take(takes, TAKE)
take.describe()

## Raw waveforms

Units are the recorded sample value (FS = digital full scale). Files are 32-bit float, so values
above 1 are legal and are **not** clipping. `highpass_hz` removes rumble so small transients are visible.

In [ ]:
fig = zp.plot_waveforms(take, T0, T1, cfg=cfg)
fig = zp.plot_waveforms(take, T0, T1, cfg=cfg, highpass_hz=1000)

## Process this take

This reads the files once, block by block, and writes the feature bundle to `cache_dir/<take>/`.
Running it again does nothing unless the files or the settings changed.

In [ ]:
za.process_take(take, cfg)
f = za.load_features(cfg, TAKE)
print("channels:", f.channels, " bands:", f.bands)
print("peak |sample| per channel:", np.round(f.meta["peak_abs"], 4))
print("samples with |x| > 1 (legal in float files, check if unexpected):", f.meta["n_samples_over_unity"])

In [ ]:
fig = zp.plot_spectrogram(f, fmax=min(24000, f.sr / 2))

In [ ]:
fig = zp.plot_levels(f, bands=[b for b in ("low", "audio", "rig") if b in f.bands], dt=1.0)

**Look for:** channels that never change (disconnected?), steps where something was moved, periodic
spikes (active chirps: mask them, see notebook 03), and the reaction onset.